In [ ]:
# ── Colab 설치 & Google Drive 마운트 ─────────────────────────────────────────
!pip install -q kiwipiepy datasets transformers scikit-learn scipy tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/vg_exp'   # 저장 경로 (원하면 수정)
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive 저장 경로: {DRIVE_DIR}')

In [ ]:
# ── Config (여기만 수정) ───────────────────────────────────────────────────────
import os
BASE_DIR     = os.path.abspath('')   # /content/
MODEL_NAME   = 'klue/roberta-base'
MAX_LEN      = 128
BATCH_SIZE   = 256
EPOCHS       = 3
LR           = 3e-5      # batch 256 기준 (warmup 6% + linear decay to 0)
WEIGHT_DECAY = 0.01
LAMBDA       = 0.1
SEEDS        = [42, 123, 456]
SUBSET       = 0         # 0 = full 172K
LAM_TARGETS  = [0.05, 0.2]  # Strict λ sensitivity ([] 로 비우면 스킵)

# 체크포인트 & 결과 → Drive에 영구 저장
CKPT_DIR    = os.path.join(DRIVE_DIR, 'checkpoints')
RESULT_PATH = os.path.join(DRIVE_DIR, 'results_final.json')
CF_PATH     = os.path.join(DRIVE_DIR, 'cf_pairs_train.jsonl')
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'CKPT_DIR    : {CKPT_DIR}')
print(f'RESULT_PATH : {RESULT_PATH}')
print(f'CF_PATH     : {CF_PATH}')

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys, json, random, gc
sys.path.insert(0, BASE_DIR)   # dataset.py가 있는 /content/ 추가
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from scipy import stats
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

from dataset import (
    SWAP_PAIRS_BY_CAT, SWAP_MAP, SWAP_KEYS,
    find_swap, make_swap,
    compute_validity, compute_validity_strict,
    load_khaters, load_cf_pairs, HatersDataset,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Model  : {MODEL_NAME}')
print(f'BASE_DIR: {BASE_DIR}')
print(f'Swap terms: {len(SWAP_KEYS)}개 ({len(SWAP_PAIRS_BY_CAT)} categories)')

In [ ]:
# ── Seed ──────────────────────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ── Model ──────────────────────────────────────────────────────────────────────
class HateDetector(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden          = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.classifier(self.dropout(cls))

    def probs(self, input_ids, attention_mask):
        return F.softmax(self.forward(input_ids, attention_mask), dim=-1)

# ── Loss ───────────────────────────────────────────────────────────────────────
def sym_kl(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    p = p.clamp(min=1e-8)
    q = q.clamp(min=1e-8)
    return (F.kl_div(q.log(), p, reduction='batchmean') +
            F.kl_div(p.log(), q, reduction='batchmean')) / 2

print('Model / Loss 정의 완료')

In [ ]:
# ── Train / Eval ───────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, use_cons: bool, lam: float):
    model.train()
    s_total = s_cls = s_cons = 0.0
    for batch in tqdm(loader, desc='  train', leave=False):
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        y     = batch['label'].to(device)
        valid = batch['cf_valid'].to(device)

        optimizer.zero_grad()
        logits   = model(ids, mask)
        cls_loss = F.cross_entropy(logits, y)
        loss     = cls_loss
        c_val    = torch.tensor(0.0, device=device)

        if use_cons and 'cf_input_ids' in batch and valid.any():
            cf_ids  = batch['cf_input_ids'].to(device)
            cf_mask = batch['cf_attention_mask'].to(device)
            p_o = model.probs(ids[valid],    mask[valid])
            p_c = model.probs(cf_ids[valid], cf_mask[valid])
            c_val = sym_kl(p_o, p_c)
            loss  = loss + lam * c_val

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        s_total += loss.item(); s_cls += cls_loss.item(); s_cons += c_val.item()

    n = len(loader)
    return s_total / n, s_cls / n, s_cons / n


def eval_f1(model, loader) -> float:
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  eval', leave=False):
            logits = model(batch['input_ids'].to(device),
                           batch['attention_mask'].to(device))
            preds.extend(logits.argmax(-1).cpu().tolist())
            labels.extend(batch['label'].tolist())
    return f1_score(labels, preds, average='macro')


def eval_fairness(model, test_examples, tokenizer):
    model.eval()
    meta = []
    for text, label, _ in test_examples:
        orig_term, swap_term, cat = find_swap(text)
        cf_text = make_swap(text, orig_term, swap_term) if orig_term else None
        meta.append((text, label, orig_term, swap_term, cat, cf_text))

    def batch_infer(texts):
        probs_all = []
        for i in range(0, len(texts), BATCH_SIZE):
            enc = tokenizer(texts[i:i+BATCH_SIZE], max_length=MAX_LEN,
                            padding='max_length', truncation=True, return_tensors='pt')
            with torch.no_grad():
                logits = model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
            probs_all.extend(F.softmax(logits, dim=-1)[:, 1].cpu().tolist())
        return probs_all

    orig_probs   = batch_infer([m[0] for m in meta])
    swap_indices = [i for i, m in enumerate(meta) if m[5] is not None]
    cf_probs_map: dict = {}
    if swap_indices:
        cf_probs_list = batch_infer([meta[i][5] for i in swap_indices])
        cf_probs_map  = dict(zip(swap_indices, cf_probs_list))

    results = []
    for i, (text, label, orig_term, swap_term, cat, cf_text) in enumerate(meta):
        prob    = orig_probs[i]
        pred    = int(prob >= 0.5)
        cf_prob = cf_probs_map.get(i)
        cf_pred = int(cf_prob >= 0.5) if cf_prob is not None else None
        results.append({'label': label, 'pred': pred, 'prob': prob,
                        'cf_pred': cf_pred, 'cf_prob': cf_prob, 'cat': cat})

    # all-swappable metrics
    swap_res      = [r for r in results if r['cf_pred'] is not None]
    flip_rate     = sum(r['pred'] != r['cf_pred'] for r in swap_res) / len(swap_res) if swap_res else 0.0
    mean_prob_gap = sum(abs(r['prob'] - r['cf_prob']) for r in swap_res) / len(swap_res) if swap_res else 0.0

    # strict-valid subset metrics
    strict_res = [
        r for r, m in zip(results, meta)
        if m[5] is not None and compute_validity_strict(m[0], m[5], m[2], m[3], m[4])['use_for_ccr']
    ]
    strict_flip_rate = sum(r['pred'] != r['cf_pred'] for r in strict_res) / len(strict_res) if strict_res else 0.0
    strict_prob_gap  = sum(abs(r['prob'] - r['cf_prob']) for r in strict_res) / len(strict_res) if strict_res else 0.0

    # FPR gap (exploratory)
    group_fp, group_tn = defaultdict(int), defaultdict(int)
    for r in results:
        if r['label'] == 0:
            grp = r['cat'] if r['cat'] else 'none'
            (group_fp if r['pred'] == 1 else group_tn)[grp] += 1
    per_group_fpr = {}
    for grp in set(list(group_fp) + list(group_tn)):
        d = group_fp[grp] + group_tn[grp]
        per_group_fpr[grp] = group_fp[grp] / d if d else 0.0
    id_fprs  = {k: v for k, v in per_group_fpr.items() if k != 'none'}
    fpr_vals = list(id_fprs.values())
    fpr_gap  = (max(fpr_vals) - min(fpr_vals)) if len(fpr_vals) >= 2 else 0.0

    return flip_rate, mean_prob_gap, strict_flip_rate, strict_prob_gap, fpr_gap, per_group_fpr


print('Train / Eval 함수 정의 완료')

In [ ]:
# ── Experiment runner ──────────────────────────────────────────────────────────
def run_experiment(tag: str, mode: str, use_cons: bool,
                   lam: float = LAMBDA, seeds=None, n_epochs: int = EPOCHS,
                   cf_lookup: dict | None = None):
    if seeds is None:
        seeds = SEEDS

    metrics = {
        'f1': [], 'flip_rate': [], 'prob_gap': [],
        'strict_flip_rate': [], 'strict_prob_gap': [],
        'fpr_gap': [], 'epoch_history': [],
    }

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    va_dl = DataLoader(
        HatersDataset(val_data, tokenizer, MAX_LEN, mode='none'),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
        pin_memory=torch.cuda.is_available())

    for seed in seeds:
        print(f'\n[{tag}] seed={seed}  lam={lam}')
        set_seed(seed)

        tr_dl = DataLoader(
            HatersDataset(train_data, tokenizer, MAX_LEN, mode=mode, cf_lookup=cf_lookup),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
            pin_memory=torch.cuda.is_available())

        model        = HateDetector(MODEL_NAME).to(device)
        opt          = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        total_steps  = len(tr_dl) * n_epochs
        warmup_steps = max(1, int(0.06 * total_steps))
        scheduler    = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)

        best_f1, best_state, seed_epochs = 0.0, {}, []
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        for ep in range(1, n_epochs + 1):
            tl, cl, cons = train_epoch(model, tr_dl, opt, scheduler, use_cons, lam)
            vf1 = eval_f1(model, va_dl)
            print(f'  ep{ep}: total={tl:.4f} cls={cl:.4f} cons={cons:.4f} | val_F1={vf1:.4f}')
            seed_epochs.append({'ep': ep, 'val_f1': round(vf1, 6),
                                 'total_loss': round(tl, 6), 'cls_loss': round(cl, 6),
                                 'cons_loss': round(cons, 6)})
            if vf1 > best_f1:
                best_f1    = vf1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                torch.save(best_state,
                           os.path.join(CKPT_DIR, f"{tag.replace(' ', '_')}_seed{seed}.pt"))

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        test_f1 = eval_f1(model, DataLoader(
            HatersDataset(test_data, tokenizer, MAX_LEN, mode='none'),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=0))
        flip, lgap, sflip, sgap, fgap, per_grp = eval_fairness(model, test_data, tokenizer)

        print(f'  test F1={test_f1:.4f}  flip={flip:.4f}  prob_gap={lgap:.4f}  '
              f'strict_flip={sflip:.4f}  strict_prob_gap={sgap:.4f}')

        metrics['f1'].append(test_f1)
        metrics['flip_rate'].append(flip)
        metrics['prob_gap'].append(lgap)
        metrics['strict_flip_rate'].append(sflip)
        metrics['strict_prob_gap'].append(sgap)
        metrics['fpr_gap'].append(fgap)
        metrics['epoch_history'].append({'seed': seed, 'epochs': seed_epochs})

        del model; gc.collect(); torch.cuda.empty_cache()

    def _s(lst): return f'{np.mean(lst):.4f}±{np.std(lst):.4f}' if lst else 'N/A'
    print(f'\n{"="*60}')
    print(f'  [{tag}]  {len(seeds)}-seed summary')
    print(f'  Macro-F1           : {_s(metrics["f1"])}')
    print(f'  Flip Rate ↓        : {_s(metrics["flip_rate"])}')
    print(f'  Prob Gap ↓         : {_s(metrics["prob_gap"])}')
    print(f'  Strict Flip Rate ↓ : {_s(metrics["strict_flip_rate"])}')
    print(f'  Strict Prob Gap ↓  : {_s(metrics["strict_prob_gap"])}')
    print(f'{"="*60}')
    return metrics


print('run_experiment 정의 완료')

---
## 데이터 로딩 & CF pair 생성

In [ ]:
# ── K-HATERS 로딩 ──────────────────────────────────────────────────────────────
print('K-HATERS 로딩 중...')
raw_train = load_khaters('train',      SUBSET)
raw_val   = load_khaters('validation', SUBSET)
raw_test  = load_khaters('test',       SUBSET)

pos_rate = sum(l for _, l, _ in raw_train) / len(raw_train)
print(f'train={len(raw_train)}  val={len(raw_val)}  test={len(raw_test)}')
print(f'train positive rate: {pos_rate:.3f}')

In [ ]:
# ── CF pair 생성 (Drive에 있으면 재사용, 없으면 생성) ─────────────────────────
if not os.path.exists(CF_PATH):
    print('CF pairs 생성 중...')
    cat_cnt, cf_pairs = Counter(), []
    for text, label, targets in raw_train:
        orig_term, swap_term, cat = find_swap(text)
        if orig_term is None:
            continue
        cat_cnt[cat] += 1
        cf_text  = make_swap(text, orig_term, swap_term)
        base_v   = compute_validity(text, cf_text, orig_term, swap_term, cat)
        strict_v = compute_validity_strict(text, cf_text, orig_term, swap_term, cat)
        cf_pairs.append({
            'original': text, 'cf': cf_text,
            'orig_term': orig_term, 'swap_term': swap_term,
            'category': cat, 'label': label, 'targets': targets,
            **{f'base_{k}': v for k, v in base_v.items()},
            **{f'strict_{k}': v for k, v in strict_v.items()},
        })
    with open(CF_PATH, 'w', encoding='utf-8') as f:
        for p in cf_pairs:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
    n_swap         = len(cf_pairs)
    n_base_valid   = sum(1 for p in cf_pairs if p['base_use_for_ccr'])
    n_strict_valid = sum(1 for p in cf_pairs if p['strict_use_for_ccr'])
    print(f'CF pairs 생성 완료 → {CF_PATH}')
    print(f'total={n_swap}, base_valid={n_base_valid}, strict_valid={n_strict_valid}')
    for cat, cnt in Counter(p['category'] for p in cf_pairs).most_common():
        print(f'  {cat}: {cnt}')
else:
    print(f'기존 CF pairs 재사용 → {CF_PATH}')

cf_lookup = load_cf_pairs(CF_PATH)
print(f'CF lookup loaded: {len(cf_lookup)} entries')

train_data, val_data, test_data = raw_train, raw_val, raw_test

---
## Main Ablation (λ=0.1, 5 methods)

In [ ]:
ABLATIONS = [
    dict(tag='Baseline',         mode='none',   use_cons=False, lam=0.0),
    dict(tag='Strict-Gated',     mode='strict', use_cons=True,  lam=LAMBDA),
    dict(tag='Naive Swap',       mode='swap',   use_cons=True,  lam=LAMBDA),
    dict(tag='Masking Cons Reg', mode='mask',   use_cons=True,  lam=LAMBDA),
    dict(tag='Validity-Gated',   mode='gated',  use_cons=True,  lam=LAMBDA),
]

all_results = {}
for exp in ABLATIONS:
    print(f"\n{'#'*60}\n  Experiment: {exp['tag']}\n{'#'*60}")
    all_results[exp['tag']] = run_experiment(**exp, cf_lookup=cf_lookup)

---
## λ Sensitivity (Strict-Gated)

In [ ]:
for lam in LAM_TARGETS:
    key = f'Strict_lam={lam}'
    print(f"\n{'#'*60}\n  Experiment: {key}\n{'#'*60}")
    all_results[key] = run_experiment(
        tag=key, mode='strict', use_cons=True, lam=lam,
        n_epochs=EPOCHS, cf_lookup=cf_lookup)

---
## Results Summary

In [ ]:
def _fmt(lst): return f'{np.mean(lst):.4f}±{np.std(lst):.4f}' if lst else 'N/A'

print('=' * 110)
print(f"  {'Model':<22} {'F1':>14} {'Flip Rate':>14} {'Prob Gap':>14} {'S-Flip':>14} {'S-ProbGap':>14}")
print('=' * 110)
for name, r in all_results.items():
    print(f"  {name:<22}  {_fmt(r['f1']):>14}  {_fmt(r['flip_rate']):>14}  "
          f"{_fmt(r['prob_gap']):>14}  {_fmt(r['strict_flip_rate']):>14}  "
          f"{_fmt(r['strict_prob_gap']):>14}")

# Paired t-test
print()
for target in ['Strict-Gated', 'Validity-Gated', 'Naive Swap', 'Masking Cons Reg']:
    if 'Baseline' in all_results and target in all_results:
        b = all_results['Baseline']['flip_rate']
        g = all_results[target]['flip_rate']
        if len(b) == len(g) and len(b) > 1:
            t, p = stats.ttest_rel(b, g)
            sig = '*significant*' if p < 0.05 else 'n.s.'
            print(f'  [t-test] Baseline vs {target:<20} '
                  f't={t:.4f}  p={p:.4f}  {sig} (α=0.05)  '
                  f'{np.mean(b):.4f}→{np.mean(g):.4f}')

# 저장
with open(RESULT_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f'\nResults saved → {RESULT_PATH}')